In [44]:
import pandas as pd
import numpy as np

In [45]:
clean_data = pd.read_parquet("../cleaned_data/clean_yellow_tripdata_2026-01.parquet")

In [46]:
df = clean_data.copy()

In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1853629 entries, 0 to 1853628
Data columns (total 35 columns):
 #   Column                        Dtype          
---  ------                        -----          
 0   vendor_id                     int8           
 1   pickup_datetime               datetime64[us] 
 2   dropoff_datetime              datetime64[us] 
 3   passenger_count               int8           
 4   trip_distance                 float32        
 5   fare_type_id                  int8           
 6   store_and_fwd_flag            boolean        
 7   pickup_location_id            int32          
 8   dropoff_location_id           int32          
 9   payment_type                  int64          
 10  fare_amount                   float64        
 11  extra_amount                  float64        
 12  mta_tax_amount                float64        
 13  tip_amount                    float64        
 14  tolls_amount                  float64        
 15  improvement_surcharge_amou

In [48]:
df.columns

Index(['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count',
       'trip_distance', 'fare_type_id', 'store_and_fwd_flag',
       'pickup_location_id', 'dropoff_location_id', 'payment_type',
       'fare_amount', 'extra_amount', 'mta_tax_amount', 'tip_amount',
       'tolls_amount', 'improvement_surcharge_amount', 'total_amount',
       'congestion_surcharge_amount', 'airport_fee_amount',
       'cbd_congestion_fee_amount', 'vendor_name', 'trip_duration',
       'taxi_speed', 'pickup_zone', 'pickup_borough', 'pickup_longitude',
       'pickup_latitude', 'dropoff_zone', 'dropoff_borough',
       'dropoff_longitude', 'dropoff_latitude', 'great_circle_distance',
       'distance_difference', 'distance_ratio', 'fare_type_zone_mismatch'],
      dtype='str')

In [49]:
cols = ['pickup_location_id', 'dropoff_location_id','pickup_zone', 'pickup_borough', 'pickup_longitude',
       'pickup_latitude', 'dropoff_zone', 'dropoff_borough','pickup_service_type',
       'dropoff_longitude', 'dropoff_latitude','dropoff_service_type']
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1853629 entries, 0 to 1853628
Data columns (total 35 columns):
 #   Column                        Dtype          
---  ------                        -----          
 0   vendor_id                     int8           
 1   pickup_datetime               datetime64[us] 
 2   dropoff_datetime              datetime64[us] 
 3   passenger_count               int8           
 4   trip_distance                 float32        
 5   fare_type_id                  int8           
 6   store_and_fwd_flag            boolean        
 7   pickup_location_id            int32          
 8   dropoff_location_id           int32          
 9   payment_type                  int64          
 10  fare_amount                   float64        
 11  extra_amount                  float64        
 12  mta_tax_amount                float64        
 13  tip_amount                    float64        
 14  tolls_amount                  float64        
 15  improvement_surcharge_amou

# change data type

In [50]:
df['pickup_location_id'] = df['pickup_location_id'].astype('int16')
df['dropoff_location_id'] = df['dropoff_location_id'].astype('int16')

# merge zone lookup file 

In [51]:
zones = pd.read_csv("../raw_data/taxi_zone_lookup.csv")

In [52]:
zones.info()

<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   LocationID    265 non-null    int64
 1   Borough       264 non-null    str  
 2   Zone          264 non-null    str  
 3   service_zone  263 non-null    str  
dtypes: int64(1), str(3)
memory usage: 17.0 KB


In [53]:
df = df.merge(zones, how='left', left_on='pickup_location_id', right_on='LocationID')
df = df.merge(zones, how='left', left_on='dropoff_location_id', right_on='LocationID')

In [54]:
drop_cols = ['pickup_borough', 'pickup_zone', 'dropoff_borough', 'dropoff_zone', 
            'LocationID_x', 'LocationID_y']
df.drop(columns=drop_cols, inplace=True)

In [55]:
df.rename(columns={
    'Borough_x' : 'pickup_borough',
    'Borough_y' : 'dropoff_borough',
    'Zone_x' : 'pickup_zone',
    'Zone_y' : 'dropoff_zone',
    'service_zone_x' : 'pickup_service_type',
    'service_zone_y' : 'dropoff_service_type'
}, inplace=True)


In [56]:
df.columns

Index(['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count',
       'trip_distance', 'fare_type_id', 'store_and_fwd_flag',
       'pickup_location_id', 'dropoff_location_id', 'payment_type',
       'fare_amount', 'extra_amount', 'mta_tax_amount', 'tip_amount',
       'tolls_amount', 'improvement_surcharge_amount', 'total_amount',
       'congestion_surcharge_amount', 'airport_fee_amount',
       'cbd_congestion_fee_amount', 'vendor_name', 'trip_duration',
       'taxi_speed', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'great_circle_distance',
       'distance_difference', 'distance_ratio', 'fare_type_zone_mismatch',
       'pickup_borough', 'pickup_zone', 'pickup_service_type',
       'dropoff_borough', 'dropoff_zone', 'dropoff_service_type'],
      dtype='str')

In [61]:
df[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 1853629 entries, 0 to 1853628
Data columns (total 12 columns):
 #   Column                Dtype  
---  ------                -----  
 0   pickup_location_id    int16  
 1   dropoff_location_id   int16  
 2   pickup_zone           str    
 3   pickup_borough        str    
 4   pickup_longitude      float32
 5   pickup_latitude       float32
 6   dropoff_zone          str    
 7   dropoff_borough       str    
 8   pickup_service_type   str    
 9   dropoff_longitude     float32
 10  dropoff_latitude      float32
 11  dropoff_service_type  str    
dtypes: float32(4), int16(2), str(6)
memory usage: 245.9 MB


# check null values of borough & zone

In [58]:
df[cols].isna().sum()

pickup_location_id          0
dropoff_location_id         0
pickup_zone              2801
pickup_borough            435
pickup_longitude         3236
pickup_latitude          3236
dropoff_zone             3240
dropoff_borough         12073
pickup_service_type      3236
dropoff_longitude       15313
dropoff_latitude        15313
dropoff_service_type    15313
dtype: int64

In [59]:
df[['pickup_zone','pickup_borough','pickup_service_type',
    'dropoff_zone','dropoff_borough','dropoff_service_type']] = df[['pickup_zone','pickup_borough','pickup_service_type',
    'dropoff_zone','dropoff_borough','dropoff_service_type']].fillna('Unknown')

In [60]:
df[cols].isna().sum()

pickup_location_id          0
dropoff_location_id         0
pickup_zone                 0
pickup_borough              0
pickup_longitude         3236
pickup_latitude          3236
dropoff_zone                0
dropoff_borough             0
pickup_service_type         0
dropoff_longitude       15313
dropoff_latitude        15313
dropoff_service_type        0
dtype: int64